In [ ]:
%matplotlib inline

# Apprentissage d'un classifieur d'image
=====================

On va voir ici comment réaliser l'apprentissage d'un CNN pour la classification d'images. On utilisera pour cela le package ``torchvision``, qui contient de nombreuses fonctions utiles pour la manipulation d'images, et permet de télécharger facilement des bases de données classiques ainsi que des modèles pré-entrainés.

Nous utiliserons le jeu de données CIFAR10, qui contient les classes ‘airplane’, ‘automobile’, ‘bird’, ‘cat’, ‘deer’, ‘dog’, ‘frog’, ‘horse’, ‘ship’, ‘truck’. Les images sont de taille 32x32 avec 3 canaux (images couleur).

![CIFAR10](https://pytorch.org/tutorials/_images/cifar10.png)



Le BE suit les étapes que l'on retrouve pour l'apprentissage de tous les réseaux de neurones, à savoir :

1. Le téléchargement et le pré-traitement des données
2. La définition d'un CNN
3. La définition d'une fonction coût (loss)
4. L'apprentissage du réseau sur le jeu de données d'entraînement (training set)
5. Le test sur le jeu de test (test set)

## 1. Chargement et pré-traitement

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

Les images d'entrée prennent des valeurs dans [0, 1], on les normalise dans l'intervalle [-1, 1], en les convertissant au passage en Tensor. On peut aussi, lors de cette étape, faire de l'augmentation de données sur les données d'entraînement. Aller voir [l'aide ici](https://pytorch.org/vision/stable/transforms.html) pour plus d'infos.

In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

torchvision.datasets.CIFAR10.url = (
    "https://huggingface.co/datasets/liangnanying/"
    "cifar-10-python/resolve/main/cifar-10-python.tar.gz"
)

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

Visualisons quelques images de l'ensemble d'entraînement



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Fonction d'affichage des images
def imshow(img):
    img = img / 2 + 0.5     # dénormalisation des images
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


# Tirage aléatoire d'images d'entraînement
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Affichage des images
imshow(torchvision.utils.make_grid(images))
# Affichage des labels associés
print(' '*7 + ' '.join(f'{classes[labels[j]]:9s}' + ' '*7 for j in range(batch_size)))

## 2. Définition du réseau (ou modèle)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # Linéarisation du tenseur avant les couches denses
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

Visualisons le réseau

In [ ]:
!pip install torchinfo

In [ ]:
from torchinfo import summary
in_data = torch.randn(batch_size, 3, 32, 32)
summary(net, input_data=in_data, col_names=['input_size', 'output_size', 'num_params'])


**Vérifier que vous comprenez bien la définition de chaque couche, en répondant aux questions suivantes :**

1. Quelle est la taille $f$ des filtres dans les deux couches de convolution?
2. Combien de filtres contient chaque couche?
3. Retrouver le nombre de paramètres de chaque couche convolutive (ne pas oublier le biais)
4. Pourquoi la taille des tenseurs diminue à chaque couche de convolution?
5. Retrouver le nombre de paramètres de chaque couche `fully connected' (*Linear*)

**Pour vérifier votre compréhension, implémentez l'architecture du réseau représenté ci-dessous :**


<center> <img src="https://acarlier.fr/tp/tctia/BE.png" width=700></center>
Le paramètre $p$ indique le *padding* utilisé (cf. cours).

## 3. Définir une fonction de coût et un optimiseur

On utilise la cross-entropy et le gradient stochastique avec momentum.

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

## 4. Entraîner le réseau



Si l'on dispose d'un GPU, on prépare l'envoi du modèle et des données sur le GPU pour accélerer l'apprentissage.

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Si la machine dispose d'un GPU compatible CUDA, ceci devrait l'afficher
print(device)

In [ ]:
net.to(device)

On réalise des "epochs", c'est-à-dire des passes sur les données. Dans chacune, on fait des itérations de descente de gradient stochastique sur des batches de données.


**Compléter le code ci-dessous.**

In [ ]:
for epoch in range(2):  # Parcours de la base d'apprentissage

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # Récupération d'un batch; data est une liste de données et labels [inputs, labels]
        # inputs, labels = data
        inputs, labels = data[0].to(device), data[1].to(device)

        # Remise à zéro des gradients des paramètres
        optimizer.zero_grad()

        ### CODE A COMPLETER
        # Passe forward : prédiction du modèle sur les entrées
        preds = ...

        # Passe backward : calcul des gradients de la fonction de coût
        # Evaluation de la fonction de coût pour le batch courant
        loss = ...
        # Calcul des gradients
        ...

        # Mise à jour des paramètres du modèle
        ...

        # Affichage de la valeur moyenne de la fonction de coût
        running_loss += loss.item()
        if i % 2000 == 1999:    # Affichage tous les 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

    # print(f'[{epoch + 1}] loss: {running_loss / i:.3f}')
    # running_loss = 0.0
print('Entraînement terminé')

**Question (à répondre ci-dessous)**


Combien semble-t-il y avoir de batches dans chaque epoch ? Expliquez ce chiffre.

On enregistre le modèle entraîné.



In [ ]:
PATH = './cifar_net.pth'
torch.save(net.state_dict(), PATH)

## 5. Test du modèle sur le jeu de données test

L'optimisation s'est bien déroulée et semble avoir à peu près convergé vers un minimum local. Mais cela ne signifie pas que le réseau a bien "appris", i.e., qu'il va pouvoir **généraliser** à de nouvelles données.

On va vérifier cela en l'appliquant sur le jeu de données de test.

In [ ]:
dataiter = iter(testloader)
images, labels = next(dataiter)

# Affichage d'images de la base de test
print('----------------------- Vérité Terrain --------------------------')
imshow(torchvision.utils.make_grid(images))
print(' '*7 + ' '.join(f'{classes[labels[j]]:9s}' + ' '*7 for j in range(batch_size)))

Si besoin on peut à présent recharger le modèle entrainé précédemment, avec la commande suivante.


In [ ]:
net = Net()
net.load_state_dict(torch.load(PATH))

Voyons à présent les prédictions de notre modèle sur quelques images du jeu de test. On rappelle que la sortie du réseau donne des probabilités d'appartenance pour chaque classe, on associe donc à chaque image la classe la plus probable.

In [ ]:
outputs = net(images)
_, predicted = torch.max(outputs, 1)


print('----------------------- Prédiction --------------------------')
imshow(torchvision.utils.make_grid(images))
print(' '*7 + ' '.join(f'{classes[predicted[j]]:9s}' + ' '*7 for j in range(batch_size)))

On effectue la même chose pour toutes les images du jeu de test.

In [ ]:
correct = 0
total = 0
# Inutile de calculer les gradients car on est en phase d'inférence
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # Prédiction du modèle sur les images du batch
        outputs = net(images)
        # La classe prédite est celle associée à la valeur maximale en sortie du réseau
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Pourcentage de bonnes classifications sur les 10000 images de test: {100 * correct // total} %')

Bonne nouvelle, on obtient un meilleur score qu'un classifieur aléatoire qui serait autour de 10%. Le réseau apprend donc vraiment. On peut regarder quelles classes posent problème :


In [ ]:
# Préparation du comptage des prédictions de chaque classe
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}


with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predictions = torch.max(outputs, 1)
        # Compte les prédictions correctes pour chaque classe
        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1


# Affichage de l'accuracy de chaque classe
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy de la classe : {classname:5s} is {accuracy:.1f} %')

## 6. Transfert d'apprentissage

Pour terminer, nous allons illustrer la notion de transfert d'apprentissage vue pendant le cours. Nous allons utiliser la base convolutive d'un modèle pré-entraîné sur une large base de données (ImageNet) pour extraire des caractéristiques avec de bonnes propriétés de généralisation. le modèle que nous allons utiliser est un ResNet-18, dont voici une vision schématique de l'architecture :    


<center> <img src="https://acarlier.fr/tp/tctia/ResNet18.png" width=900></center>


En affichant la variable *rn_model*, on observe la structure du réseau dont les différents blocs et couches sont accessibles par leur nom. Le bloc représenté en bleu sur la figure s'appelle *layer1*, le bloc vert *layer2*, etc. La dernière couche (*fc*) est une couche complètement connectée (*Linear*) prenant en entrée un vecteur de caractéristiques de dimension 512 et renvoyant un vecteur de taille 1000, correspondant aux probabilités de classe associées aux 1000 classes de la base de données ImageNet.



In [ ]:
# On utilise comme base convolutive un ResNet 18 pré-entraîné sur ImageNet
rn_model = models.resnet18(weights='IMAGENET1K_V1')
print(rn_model)

# Affichage détaillé avec torchsummary
in_data = torch.randn(batch_size, 3, 224, 224)
summary(rn_model, input_data=in_data, col_names=['input_size', 'output_size', 'num_params'])

Nous voulons remplacer cette dernière couche par  une couche adaptée à la base de données CIFAR-10, avec seulement 10 neurones de sortie (pour les 10 classes de CIFAR-10).  Dans un premier temps, nous allons geler tous les paramètres de la base convolutive (représentée en orange ci-dessous) et entraîner seulement cette dernière couche (représentée en vert ci-dessous) que nous allons modifier.


<center> <img src="https://acarlier.fr/tp/tctia/ResNet18_TL.png" width=900></center>


In [ ]:
# Pour geler les paramètres, on positionne la variable requires_grad à False pour toutes les couches concernées
for param in rn_model.parameters():
  param.requires_grad = False

On définit la nouvelle couche de sortie du réseau

In [ ]:
### A COMPLETER
rn_model.fc = ...

**Question** : combien de paramètres allons-nous entraîner avec cette version du réseau ?

Avant l'apprentissage, il faut également redéfinir les *DataLoader* qui effectuent le chargement des données. En effet, le modèle ResNet18 que nous utilisons attend un pré-traitement des données particulier décrit dans la [documentation](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet18.html).

In [ ]:
transform = transforms.Compose(
    [transforms.Resize(224), # Mise à l'échelle des images en 224 x 224 x 3
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], # Normalisation attendue par ResNet18
                         [0.229, 0.224, 0.225])])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

On peut maintenant réutiliser notre code précédent pour entraîner ce modèle sur CIFAR-10 :

In [ ]:
### A COMPLETER
### Entraînez ce nouveau modèle pendant 2 epochs et évaluez les résultats sur l'ensemble de test
...

A ce stade les résultats sont comparables à ceux obtenus en entraînant notre propre modèle, mais gardez à l'esprit que le nombre de paramètres que nous avons entraîné dans le réseau Resnet-18 est très réduit. De plus, les images CIFAR-10, initialement de dimension $32 \times 32$, ont été redimensionnées en $224 \times 224$, ce qui crée des artefacts visuels importants qui n'existent pas dans la base de données ImageNet. La base convolutive de Resnet-18 n'a pas été entraînée sur des images comportant ces artefacts et les filtres de convolution ne sont certainement pas les plus adaptés pour bien reconnaître les images CIFAR-10.



## Fine-tuning

Une solution pour améliorer les résultats consiste à autoriser l'entraînement d'un plus grand nombre de couches du réseau ResNet-18. Nous allons ainsi dégeler les paramètres du bloc *layer4*, et entraîner toutes les couches représentées en vert ci-dessous :     

<center> <img src="https://acarlier.fr/tp/tctia/ResNet18_FT.png" width=900></center>


In [ ]:
### A COMPLETER pour autoriser l'entraînement des couches du bloc 4
for param in rn_model.layer4.parameters():
  ...

In [ ]:
### A COMPLETER
### Entraînez ce nouveau modèle pendant 2 epochs et évaluez les résultats sur l'ensemble de test
...

Cette fois-ci les résultats devraient s'être beaucoup améliorés !

## 7. Transformer le CNN en réseau entièrement convolutif

Jusqu'à présent, nous avons construit un réseau de neurones convolutif destiné à la classification d'images. Ce réseau prend en entrée une image de dimension 32 × 32 pixels et produit un vecteur de 10 valeurs correspondant aux scores associés aux différentes classes de CIFAR-10.

Une limitation de notre architecture vient de la présence des couches fully connected (*Linear*), qui imposent une dimension fixe à l'entrée du réseau.

Nous allons maintenant modifier cette architecture pour obtenir un réseau entièrement convolutif (*Fully Convolutional Network*, FCN).

L'idée consiste à remplacer les couches fully connected par des couches convolutionnelles. Cela permettra notamment d'appliquer le réseau à des images de dimensions différentes de celles utilisées pendant l'apprentissage.


Compléter la définition de la classe `FCN` en remplaçant les couches fully connected du réseau `Net` par des couches convolutionnelles adaptées.

Conserver les mêmes fonctions d'activation et les mêmes opérations de pooling.

In [ ]:
class FCN(nn.Module):
    def __init__(self):
        super().__init__()


    def forward(self, x):


fcn = FCN()


Pour vérifier que la transformation conserve bien le comportement du réseau initial, nous allons transférer les poids appris précédemment.

En effet, une couche fully connected peut être convertie en couche convolutionnelle en réorganisant simplement ses paramètres.


In [ ]:
# Transfert des poids des couches convolutives
fcn.conv1.load_state_dict(net.conv1.state_dict())

## A COMPLETER

# Ici on suppose la la conv4 de votre FCN correspond à la première couche dense de votre réseau précédent
fcn.conv4.weight.data = net.fc1.weight.data.reshape(...) # INDIQUER LES BONNES DIMENSIONS
fcn.conv4.bias.data = net.fc1.bias.data.clone()

## A COMPLETER



Vérifions maintenant que les deux réseaux produisent les mêmes résultats sur une image de CIFAR-10.

In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

torchvision.datasets.CIFAR10.url = (
    "https://huggingface.co/datasets/liangnanying/"
    "cifar-10-python/resolve/main/cifar-10-python.tar.gz"
)

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
net.eval()
fcn.eval()

images, labels = next(iter(testloader))

images = images.to(device)

net = net.to(device)
fcn = fcn.to(device)

with torch.no_grad():
    output_net = net(images)
    output_fcn = fcn(images)

print("Dimension de sortie du CNN :", output_net.shape)
print("Dimension de sortie du FCN :", output_fcn.shape)

# Comparaison des sorties
print("Erreur maximale :",
      torch.max(torch.abs(output_net - output_fcn.squeeze(-1).squeeze(-1))).item())

## 9. Application à une image de grande dimension

Jusqu'à présent, nous avons utilisé le réseau sur des images de 32 × 32 pixels.

Grâce à sa structure entièrement convolutive, nous pouvons maintenant lui fournir des images de dimensions beaucoup plus importantes.

Nous allons appliquer notre réseau à une image plus grande, par exemple de 96 × 96 pixels.

Contrairement au CNN initial, le FCN ne produira plus un unique vecteur de 10 scores, mais une carte spatiale de scores pour chacune des dix classes.

Chaque position de cette carte correspond à une région de l'image d'entrée.


In [ ]:
import requests
from io import BytesIO
from PIL import Image

# Image distante (vous pouvez changer l'URL pour tester vos propres images)
response = requests.get("https://media.easy-peasy.ai/6028d3f4-6eba-4d8f-a7b8-627afb3f63d6/098a02cd-a5cb-43bc-9b1a-c08628f6c15d_medium.webp")

# Chargement de l'image
image = Image.open(BytesIO(response.content)).convert("RGB")

# Redimensionnement à 256 x 256 pixels
transform_large = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])

image_tensor = transform_large(image).unsqueeze(0)
print("Dimension de l'image :", image_tensor.shape)

plt.figure(figsize=(6, 6))
img = image_tensor.squeeze(0).cpu()

# Dénormalisation
img = img * 0.5 + 0.5
img = img.permute(1, 2, 0).numpy()

plt.imshow(img)
plt.axis("off")
plt.title("Image d'entrée")
plt.show()

Nous pouvons maintenant appliquer le réseau à cette image.

Contrairement à ce que nous avons fait précédemment, la sortie du réseau possède maintenant deux dimensions spatiales.

À chaque position, nous disposons de dix scores correspondant aux dix classes de CIFAR-10.

Nous allons sélectionner la classe associée au score maximal.

In [ ]:
fcn.eval()

with torch.no_grad():
    outputs = fcn(image_tensor.to(device))

print("Dimension de sortie :", outputs.shape)

# Classe prédite à chaque position spatiale
prediction = outputs.argmax(dim=1)

print("Dimension de la carte de prédictions :", prediction.shape)

Utilisez maintenant le code suivant pour afficher l'image et la carte des classes prédites pour chaque pixel de l'image. Ce résultat est une segmentation très grossière de l'image, que l'on pourra affiner en complétant le modèle avec une partie décodeur, pour retrouver une résolution plus fine.

In [ ]:
def display_image(img_tensor, ax=None):
    img = img_tensor.squeeze(0).cpu()

    # Dénormalisation
    img = img * 0.5 + 0.5

    img = img.permute(1, 2, 0).numpy()

    if ax is None:
        plt.imshow(img)
        plt.axis("off")
    else:
        ax.imshow(img)
        ax.axis("off")


# Redimensionnement de la carte de prédictions
prediction_large = F.interpolate(
    prediction.unsqueeze(1).float(),
    size=(96, 96),
    mode="nearest"
)

prediction_large = prediction_large.squeeze().cpu().numpy().astype(int)

# Affichage
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

display_image(image_tensor, axes[0])
axes[0].set_title("Image originale")

im = axes[1].imshow(
    prediction_large,
    cmap="tab10",
    vmin=0,
    vmax=9,
    interpolation="nearest"
)

axes[1].set_title("Carte des classes prédites")

cbar = plt.colorbar(im, ax=axes[1], ticks=range(10))
cbar.ax.set_yticklabels(classes)

plt.tight_layout()
plt.show()